In [1]:
# Cell 1: Import necessary libraries

import os
import numpy as np
import time
from pydrake.all import (
    DiagramBuilder, AddMultibodyPlantSceneGraph, Parser,
    Role, MeshcatVisualizer, StartMeshcat, RationalForwardKinematics, CspaceFreePolytope,
    SeparatingPlaneOrder, Rgba, RollPitchYaw,
    LinearEqualityConstraint, Sphere, Parallelism, AddDefaultVisualization, 
    ConnectPlanarSceneGraphVisualizer, IrisFromCliqueCoverOptions, 
    CommonSampledIrisOptions, IrisZoOptions, IrisNp2Options,
    IrisInConfigurationSpaceFromCliqueCover, RandomGenerator, RobotDiagramBuilder, 
    SceneGraphCollisionChecker, MultibodyPlant, SceneGraph, 
    SolverOptions, CommonSolverOption, GeometrySet, ScsSolver
)
from pydrake.geometry.optimization import GraphOfConvexSetsOptions, HPolyhedron, VPolytope, Point, Hyperellipsoid
from pydrake.geometry.optimization import ConvexHull as DrakeConvexHull
from pydrake.planning import GcsTrajectoryOptimization
from pydrake.solvers import MathematicalProgram, Solve, MosekSolver
from pydrake.trajectories import CompositeTrajectory
from pydrake.common import FindResourceOrThrow
from scipy.spatial import ConvexHull
from pydrake.math import RigidTransform, RotationMatrix
from pydrake.multibody.inverse_kinematics import InverseKinematics
from pydrake.solvers import Solve, SnoptSolver, IpoptSolver, OsqpSolver, ScsSolver
import mcubes
from functools import partial
import matplotlib.pyplot as plt
from ipywidgets import widgets
import quadprog

from pathlib import Path
import sys

# add the parent directory of this notebook to the import path
parent = Path.cwd().parent
if str(parent) not in sys.path:
    sys.path.insert(0, str(parent))

from ciris_plant_visualizer import CIrisPlantVisualizer

In [2]:
!pip show drake

Name: drake
Version: 1.48.0
Summary: Model-based design and verification for robotics
Home-page: https://drake.mit.edu
Author: Drake Development Team
Author-email: drake-users@mit.edu
License: Various
Location: /home/julialopezgomez/miniforge3/envs/obmp/lib/python3.13/site-packages
Requires: matplotlib, Mosek, numpy, pydot, PyYAML
Required-by: 


In [7]:
# Cell 2: Set up the plant and scene graph, and initialize the CIrisPlantVisualizer
# builder = DiagramBuilder()
# plant, scene_graph = AddMultibodyPlantSceneGraph(builder, time_step=0.0)
# parser = Parser(plant, scene_graph)
# parser.SetAutoRenaming(True)

print("Setting up the plant and scene graph...")

# Replace DiagramBuilder with RobotDiagramBuilder
builder = RobotDiagramBuilder(time_step=0.0)
plant = builder.plant()
scene_graph = builder.scene_graph()
parser = Parser(plant, scene_graph)
parser.SetAutoRenaming(True)

Setting up the plant and scene graph...


In [8]:
# Add the robot
# gripper = parser.AddModels(file_name="../my_sdfs/wsg_2dof.sdf")[0]

print("Loading Panda robot models...")

# --- Add the Panda arm + hand ---
panda_arm  = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_arm.urdf")[0]
panda_hand = parser.AddModels(url="package://drake_models/franka_description/urdf/panda_hand.urdf")[0]
# panda_arm  = parser.AddModels(url="file:///home/julialopezgomez/optimisation-based-manipulation-planner/my_sdfs/panda_arm_locked.urdf")[0]
# panda_hand = parser.AddModels(url="file:///home/julialopezgomez/optimisation-based-manipulation-planner/my_sdfs/panda_hand_locked.urdf")[0]

print("Welding Panda arm and hand...")

# Weld arm base to world (identity)
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("panda_link0", panda_arm),
    RigidTransform())

# Weld hand to arm flange with X_PC: translation [0,0,0], RPY deg [0,0,-45]
X_8H = RigidTransform(RollPitchYaw(0.0, 0.0, -np.deg2rad(45.0)), [0.0, 0.0, 0.0])
plant.WeldFrames(
    plant.GetFrameByName("panda_link8", panda_arm),        # parent (P)
    plant.GetFrameByName("panda_hand", panda_hand),    # child  (C)
    X_8H)

print("Setting default finger joint positions...")

# Optional: set the default finger opening (each finger is a prismatic joint).
# 0.02 m on each finger → ~0.04 m total width. Adjust to taste.
for j in ["panda_finger_joint1"]:#, "panda_finger_joint2"]:
    plant.GetJointByName(j, panda_hand).set_default_translation(-0.024)
    

    


Loading Panda robot models...
Welding Panda arm and hand...
Setting default finger joint positions...


In [9]:
print("Adding the bottle cap and obstacles...")

cap = parser.AddModels(file_name="../my_sdfs/bottle_cap.sdf")[0]
obstacle1 = parser.AddModels("../my_sdfs/obstacle.sdf")[0]
# obstacle2 = parser.AddModels("my_sdfs/obstacle.sdf")[0]
obstacle3 = parser.AddModels("../my_sdfs/obstacle.sdf")[0]

# Set welds
plant.WeldFrames(
    plant.world_frame(), 
    plant.GetFrameByName("base_link", cap),
    RigidTransform(RotationMatrix(), [0.5, 0, 0]))

# Weld the obstacle to the world frame (adjust pose as needed)
obstacle_pose1 = RigidTransform(RotationMatrix(), [0.51, 0.031, 0.01])  # Adjust position
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle1),
    obstacle_pose1)

# # obstacle_pose2 = RigidTransform(RotationMatrix(), [-0.025, 0.05, 0.01])  # Adjust position
# # plant.WeldFrames(
# #     plant.world_frame(),
# #     plant.GetFrameByName("obstacle_link", obstacle2),
# #     obstacle_pose2)

obstacle_pose3 = RigidTransform(RotationMatrix(), [0.467, -0.005, 0.01])  # Adjust position
plant.WeldFrames(
    plant.world_frame(),
    plant.GetFrameByName("obstacle_link", obstacle3),
    obstacle_pose3)

Adding the bottle cap and obstacles...


<WeldJoint name='world_welds_to_obstacle_link' index=16 model_instance=6>

In [10]:
print("Finalising plant...")
print("Bodies:", plant.num_bodies(), "Joints:", plant.num_joints())


plant.Finalize()

inspector = scene_graph.model_inspector()
num_frames = len(list(inspector.GetAllFrameIds()))
num_geoms = sum(inspector.NumGeometriesForFrame(fid) for fid in inspector.GetAllFrameIds())
print("Frames:", num_frames, "Geometries:", num_geoms)


print("Plant finalised.")

print("Number of positions: ", plant.num_positions())

# Cell 3: Initialize the CIrisPlantVisualizer
q_star = np.zeros(plant.num_positions())


print("Initialising CspaceFreePolytope...")

# The object we will use to perform our certification
cspace_free_polytope = CspaceFreePolytope(
    plant, 
    scene_graph,
    SeparatingPlaneOrder.kAffine,
    q_star)

print("Initializing CIrisPlantVisualizer...")

visualizer = CIrisPlantVisualizer(
    plant,
    builder,
    scene_graph,
    cspace_free_polytope,
    viz_role=Role.kIllustration,
    # viz_role=Role.kProximity,
    allow_plus_3dof=True
)

print("Setting up the visualizer...")

visualizer.task_space_diagram.ForcedPublish(visualizer.task_space_diagram_context)


INFO:drake:Meshcat listening for connections at http://localhost:7001


Finalising plant...
Bodies: 18 Joints: 17
Frames: 18 Geometries: 94
Plant finalised.
Number of positions:  10
Initialising CspaceFreePolytope...
Initializing CIrisPlantVisualizer...
Visualisations won't work properly. Can't visualize the TC-Space of plants with more than 3-DOF. The first 3 DOF from the plant will be visualized
Setting up the visualizer...


In [12]:

sliders = []

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

for i in range(plant.num_positions()):
    q_low = plant.GetPositionLowerLimits()[i]
    q_high = plant.GetPositionUpperLimits()[i]
    step = (q_high - q_low) / 100
    sliders.append(widgets.FloatSlider(
        min=q_low, max=q_high, 
        value=0, step=step, 
        description=f"q{i}"))
    
q = np.zeros(plant.num_positions())

def report_collisions(q):
    plant.SetPositions(plant_context, q)
    query_object = scene_graph.get_query_output_port().Eval(
        scene_graph.GetMyContextFromRoot(diagram_context)
    )
    inspector = scene_graph.model_inspector()

    pairs = query_object.ComputePointPairPenetration()
    if not pairs:
        print("No penetrations")
        return

    for pair in pairs:
        name_A = inspector.GetName(pair.id_A)
        name_B = inspector.GetName(pair.id_B)
        print(f"{name_A} <-> {name_B}, depth={pair.depth}")



out = widgets.Output()
display(out)

def handle_slider_change(change, idx):
    q[idx] = change["new"]
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)
    with out:
        out.clear_output(wait=True)
        
        report_collisions(q)
        print(f"{visualizer.check_collision_q_by_ik(q)} \t {q}", flush=True)
    
    
idx = 0
for slider in sliders:
    slider.observe(partial(handle_slider_change, idx = idx), names='value')
    idx+=1

for slider in sliders:
    display(slider)

Output()

FloatSlider(value=0.0, description='q0', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q1', max=1.7628, min=-1.7628, step=0.035255999999999996)

FloatSlider(value=0.0, description='q2', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=-0.0698, description='q3', max=-0.0698, min=-3.0718, step=0.03002)

...

FloatSlider(value=0.0, description='q5', max=3.7525, min=-0.0175, step=0.0377)

FloatSlider(value=0.0, description='q6', max=2.8973, min=-2.8973, step=0.057946)

FloatSlider(value=0.0, description='q7', max=0.0, min=-0.045, step=0.00045)

FloatSlider(value=0.0, description='q8', max=0.045, step=0.00045)

FloatSlider(value=0.0, description='q9', max=3.14, min=-3.14, step=0.06280000000000001)

In [ ]:
def joint_position_index(plant, joint_name, model_instance=None):
    if model_instance is None:
        joint = plant.GetJointByName(joint_name)
    else:
        joint = plant.GetJointByName(joint_name, model_instance)

    if joint.num_positions() != 1:
        raise ValueError(
            f"Joint {joint_name} has {joint.num_positions()} positions; "
            "this helper assumes a 1-DOF joint."
        )

    return joint.position_start()

for i, name in enumerate(plant.GetPositionNames()):
    print(f"{i:2d}: {name:40s}")



 0: panda_panda_joint1_q                    
 1: panda_panda_joint2_q                    
 2: panda_panda_joint3_q                    
 3: panda_panda_joint4_q                    
 4: panda_panda_joint5_q                    
 5: panda_panda_joint6_q                    
 6: panda_panda_joint7_q                    
 7: panda_hand_panda_finger_joint1_x        
 8: panda_hand_panda_finger_joint2_x        
 9: bottle_cap_cap_to_base_q                


## Find Grasping configuration

In [14]:
def solve_ik_place_frame_at_pose(
    # visualizer,
    plant,
    plant_context,
    frame_B,                                # the frame you want to place (e.g., panda_hand)
    X_WG: RigidTransform,                   # desired pose of frame_B in world
    pos_tol=1e-3,                           # box half-width [m]
    theta_tol=np.deg2rad(2),                # angular tolerance [rad]
    min_distance=None,                      # e.g., 1e-3 to keep clearance; None to ignore
    q_seed=None,                            # seed configuration
):
    
    ik = InverseKinematics(plant, plant_context)
    q = ik.q()

    # Position: keep the origin of frame_B in a small box around X_WG.translation()
    p_BQ_B = np.zeros(3)
    p = X_WG.translation()
    ik.AddPositionConstraint(
        frameB=frame_B, p_BQ=p_BQ_B,
        frameA=plant.world_frame(),
        p_AQ_lower=p - pos_tol, p_AQ_upper=p + pos_tol
    )

    # Orientation: bound frame_B’s rotation to the desired world rotation
    # This constrains angle between R_WB and X_WG.rotation() to be <= theta_tol.    
    ik.AddOrientationConstraint(
    frameAbar=plant.world_frame(),            # Ā
    R_AbarA=X_WG.rotation(),                  # desired world rotation
    frameBbar=frame_B,                        # B̄ = your hand frame
    R_BbarB=RotationMatrix(),                 # identity in B
    theta_bound=theta_tol                     # radians
)


    # Optional collision clearance (uses Proximity role; can be conservative for grasps)
    if min_distance is not None:
        ik.AddMinimumDistanceLowerBoundConstraint(min_distance)

    prog = ik.prog()
    if q_seed is None:
        q_seed = plant.GetPositions(plant_context)
    prog.SetInitialGuess(q, q_seed)

    # Pick an available solver. Orientation makes this nonlinear; SNOPT/IPOPT are best.
    result = Solve(prog)
    if not result.is_success():
        # try a second pass with a looser theta or different seed before giving up
        return None, result

    q_sol = result.GetSolution(q)

    return q_sol, result


In [15]:
# Build diagram/contexts as you already do
plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

# Frames and goal pose
E   = plant.GetFrameByName("panda_hand", panda_hand)   # tool frame
Cap = plant.GetFrameByName("base_link", cap)           # cap frame

# World poses
X_WCap = plant.CalcRelativeTransform(plant_context, plant.world_frame(), Cap)

# Choose a grasp goal: 65 mm above cap origin (same orientation as Cap here)
# Rotate by 180 deg upside down
R_CapGoal = RotationMatrix(RollPitchYaw(np.pi, 0, 0))
X_CapGoal = RigidTransform(R_CapGoal, [0.0, 0.0, 0.105])
X_WG = X_WCap.multiply(X_CapGoal)  # desired world pose for the hand frame

# (Optional) lock fingers during IK
for jn in ["panda_finger_joint1", "panda_finger_joint2"]:
    plant.GetJointByName(jn, panda_hand).Lock(plant_context)

# Solve IK (function from previous message)
q_sol, res = solve_ik_place_frame_at_pose(
    # visualizer=visualizer,
    plant=plant,
    plant_context=plant_context,
    frame_B=E,
    X_WG=X_WG,
    pos_tol=0.0,
    theta_tol=np.deg2rad(0),
    min_distance=None,
    q_seed=plant.GetPositions(plant_context)
)

if q_sol is None:
    print("IK failed:", res.get_solver_id().name())
else:
    plant.SetPositions(plant_context, q_sol)
    diagram.ForcedPublish(diagram_context)
    print("IK succeeded. New q:", q_sol)

IK succeeded. New q: [ 1.37476094e+00  1.56023543e+00 -1.28753190e+00 -2.21771480e+00
  1.73595107e+00  1.80222917e+00 -7.52900343e-02  0.00000000e+00
  0.00000000e+00  8.88178420e-16]


## Grasping Configuration checker

Hand frame must be located at a certain height range from the cap (and rotation, so a transform), fingers at an offset (also a range) and allow full rotation of gripper hand and cap

In [16]:
# Get panda_hand frame between 0.0105 m and 0.11 m above the cap base_link frame
# Fingers should be open between 0.024 and 0.025 m and -0.024 and -0.025 m
# Get a function that checks whether these constraints are satisfied

def check_grasp_constraints(q):
    old_q = plant.GetPositions(plant_context)
    plant.SetPositions(plant_context, q)
    # diagram.ForcedPublish(diagram_context)
    
    # Get the current pose of the hand frame
    X_WE = plant.CalcRelativeTransform(plant_context, plant.world_frame(), E)
    X_WCap = plant.CalcRelativeTransform(plant_context, plant.world_frame(), Cap)
    
    # Compute the relative transform from Cap to E
    X_CapE = X_WCap.inverse().multiply(X_WE)
    
    z_height = X_CapE.translation()[2]
    
    right_finger_joint = plant.GetJointByName("panda_finger_joint1", panda_hand)
    left_finger_joint  = plant.GetJointByName("panda_finger_joint2", panda_hand) # Uncomment this line if you want to check the left finger position as well
    
    right_finger_pos = right_finger_joint.get_translation(plant_context)
    left_finger_pos  = left_finger_joint.get_translation(plant_context) # Uncomment this line if you want to check the left finger position as well
    
    # Check height constraint
    height_ok = 0.0105 <= z_height <= 0.11
    
    # Check finger opening constraints
    fingers_ok = (-0.025 <= right_finger_pos <= -0.024) and (0.024 <= left_finger_pos <= 0.025)
    
    # Restore old q
    plant.SetPositions(plant_context, old_q)
    diagram.ForcedPublish(diagram_context)
    
    print(f"Height: {z_height:.4f} m, Right Finger: {right_finger_pos:.4f} m, Left Finger: {left_finger_pos:.4f} m, Height OK: {height_ok}, Fingers OK: {fingers_ok}")
    
    return height_ok and fingers_ok, z_height, right_finger_pos #, left_finger_pos

In [17]:

plant_context = visualizer.plant_context
diagram = visualizer.task_space_diagram
diagram_context = visualizer.task_space_diagram_context

# Frames and goal pose
E   = plant.GetFrameByName("panda_hand", panda_hand)   # tool frame
Cap = plant.GetFrameByName("base_link", cap)           # cap frame

# World poses
X_WCap = plant.CalcRelativeTransform(plant_context, plant.world_frame(), Cap)

# Choose a grasp goal: 65 mm above cap origin (same orientation as Cap here)
# Rotate by 180 deg upside down
R_CapGoal = RotationMatrix(RollPitchYaw(np.pi, 0, 0))
X_CapGoal = RigidTransform(R_CapGoal, [0.0, 0.0, 0.105])
X_WG = X_WCap.multiply(X_CapGoal)  # desired world pose for the hand frame

# Set joints to grasping position:
q_sol[7] = -0.024
q_sol[8] = 0.024

plant.SetPositions(plant_context, q_sol)

diagram.ForcedPublish(diagram_context)

print("Checking grasp constraints for q_grasp:", check_grasp_constraints(q_sol))



Height: 0.1050 m, Right Finger: -0.0240 m, Left Finger: 0.0240 m, Height OK: True, Fingers OK: True
Checking grasp constraints for q_grasp: (True, np.float64(0.10500000000005476), -0.024)


## 1.2. Setup Grasping and Placement Space

In [18]:
builder = visualizer.builder
plant = visualizer.plant
scene_graph = visualizer.scene_graph
q_star = visualizer.q_star
rat_fk = visualizer.rat_forward_kin
inspector = visualizer.model_inspector
diagram = visualizer.task_space_diagram
context = visualizer.task_space_diagram_context
cspace_free_polytope = visualizer.cspace_free_polytope

In [19]:
# Define the 8 corner points of the convex hull
z_bounds = [-3.14, 3.14]
x_bounds = [-1, 1]
y_bounds = [-0.045, -0.024]

lower_joint_limits = np.array([x_bounds[0], y_bounds[0], z_bounds[0]])
upper_joint_limits = np.array([x_bounds[1], y_bounds[1], z_bounds[1]])

y_bounds_grasp = [-0.025, -0.024]

# Generate all corner points
placement_points = np.array([[x, y, z] for x in x_bounds for y in y_bounds for z in z_bounds])
grasp_points = np.array([[x, y, z] for x in x_bounds for y in y_bounds_grasp for z in z_bounds])
###### For 2 dimensional example:
# placement_points = np.array([[x, y] for x in x_bounds for y in z_bounds])
# grasp_points = np.array([[x, y] for x in x_bounds for y in z_bounds_grasp])

# Compute the convex hull
placement_hull = ConvexHull(placement_points)
grasp_hull = ConvexHull(grasp_points)

# Convert ConvexHull to HPolyhedron
def convex_hull_to_hpolyhedron(hull):
    A = hull.equations[:, :-1]
    b = -hull.equations[:, -1]
    return HPolyhedron(A, b)

placement_polytope = convex_hull_to_hpolyhedron(placement_hull)
grasp_polytope = convex_hull_to_hpolyhedron(grasp_hull)


In [20]:
print(visualizer.q_lower_limits)
print(visualizer.q_upper_limits)

[-2.8973 -1.7628 -2.8973 -3.0718 -2.8973 -0.0175 -2.8973 -0.045   0.
 -3.14  ]
[ 2.8973  1.7628  2.8973 -0.0698  2.8973  3.7525  2.8973  0.      0.045
  3.14  ]


# Manipulation Planning Class

In [ ]:
from pydrake.geometry.optimization import LoadIrisRegionsYamlFile


class ManipulationPlanner():

    def __init__(self, 
            visualizer: CIrisPlantVisualizer,
            CP: HPolyhedron,
            CG: HPolyhedron,
            max_grasps: int = 20,
            model_instances: list = None,
            gripper_dim: int = None,
            cs_free: list[HPolyhedron] = None,
            irisalg: str = "irisnp"
        ):
        self.builder = visualizer.builder
        self.plant = visualizer.plant
        self.plant_context = visualizer.plant_context
        self.scene_graph = visualizer.scene_graph
        self.q_star = visualizer.q_star
        self.rat_fk = visualizer.rat_forward_kin
        self.inspector = visualizer.model_inspector
        self.diagram = visualizer.task_space_diagram
        self.diagram_context = visualizer.task_space_diagram_context
        self.lower_joint_limits = visualizer.q_lower_limits
        self.upper_joint_limits = visualizer.q_upper_limits
        self.visualize_cspace = visualizer.visualize_collision_constraint
        self.cspace_free_polytope = visualizer.cspace_free_polytope
        
        self.CP = CP
        self.CG = CG
        self.max_grasps = max_grasps

        self.model_instances = model_instances if model_instances is not None else [self.plant.GetModelInstanceByName("robot")]
        
        self.q_dim = self.plant.num_positions()
        self.gripper_dim = gripper_dim if gripper_dim is not None else self.q_dim
        
        
        self.cs_free = cs_free if cs_free is not None else self._generate_cfree(irisalg=irisalg)
    
        
        
        os.environ["MOSEKLM_LICENSE_FILE"] = "mosek.lic"
        with open(os.environ["MOSEKLM_LICENSE_FILE"], 'r') as f:
            contents = f.read()
            mosek_file_not_empty = contents != ''
                    
        assert mosek_file_not_empty, "Mosek license file is empty. Please ensure you have a valid license file at the path specified by MOSEKLM_LICENSE_FILE environment variable."
        assert MosekSolver().available(), "Mosek solver is not available. Please ensure you have Mosek installed and properly configured."
        
    def set_max_grasps(self, max_grasps: int):
        self.max_grasps = max_grasps
    
    
    def compute_trajectory(self, x_init, x_goal, display=True):
        print("Finding path for minimum grasps")
        path = self._find_minimum_grasp_path(
                x_init = x_init,
                x_goal = x_goal,
                P = self.CP,
                G = self.CG,
                lower_joint_limits = self.lower_joint_limits,
                upper_joint_limits = self.upper_joint_limits,
                c_free_polytopes = self.cs_free
            )
        
        print("Path found. Generating trajectory")
        traj = self._generate_trajectory(path)
        
        
        print("Trajectory generated. Starting display")
        if display:
            self.display_trajectory(traj)
            
        return path, traj
        
    def display_trajectory(self, traj, meshcat=True, plotly=False):
        if meshcat:
            num_points = int((traj.end_time() - traj.start_time()) * 4000)
            for t in np.linspace(traj.start_time(), traj.end_time(), num_points):
                q = traj.value(t)
                # self.plant.GetJointByName("left_finger_sliding_joint", gripper).set_translation(self.plant_context, q[2])
                # # self.plant.GetJointByName("right_finger_sliding_joint", gripper).set_translation(self.plant_context, q[3])
                # self.plant.GetJointByName("base_revolute_joint", gripper).set_angle(self.plant_context, q[1])
                # self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[0])
                self.plant.GetJointByName("panda_finger_joint1", panda_hand).set_translation(self.plant_context, q[1][0])
                self.plant.GetJointByName("panda_joint7", panda_arm).set_angle(self.plant_context, q[0][0])
                self.plant.GetJointByName("cap_to_base", cap).set_angle(self.plant_context, -q[2][0])
                self.diagram.ForcedPublish(self.diagram_context)
                time.sleep(0.01)
        path = [traj.value(t) for t in np.linspace(traj.start_time(), traj.end_time(), 100)]
        path_q = [q.ravel() for q in path] # Convert to list of numpy arrays
        if plotly:
            self.visualize_cspace(factor=1, num_points=30, paths=[path_q], filled_polytopes=self.cs_free)
        return path_q
    
    def _generate_trajectory(self, path):
        trajs = []
        print("Generating trajectory from:")
        for q0, q1 in zip(path[:-1], path[1:]):
            print("From\t{}\tto\t{}".format(q0, q1))
            traj, result = self._generate_edge_trajectory(q0, q1)
            if not result.is_success():
                print("failed to generate edge trajectory from {} to {}".format(q0, q1))
                return None
            trajs.append(traj)
        return CompositeTrajectory.AlignAndConcatenate(trajs)
        
        
    def _generate_edge_trajectory(self, x_init, x_goal, path_length_weight=None):
        trajopt = GcsTrajectoryOptimization(self.plant.num_positions())
        gcs_regions = trajopt.AddRegions(self.cs_free, order=1, h_min=0.01)
        source = trajopt.AddRegions([Point(x_init)], order=0)
        target = trajopt.AddRegions([Point(x_goal)], order=0)
        trajopt.AddEdges(source, gcs_regions)
        trajopt.AddEdges(gcs_regions, target)
        trajopt.AddPathLengthCost(path_length_weight if path_length_weight is not None else 1.0)
        options = GraphOfConvexSetsOptions()
        [traj, result] = trajopt.SolvePath(source, target, options)
        print(f"result.is_success() = {result.is_success()}")
        print(f"result.get_solution_result() = {result.get_solution_result()}")
        print(f"result.get_solver_id().name() = {result.get_solver_id().name()}")
        
        
        # print("\n--- GCS RESULT ---")
        # print("success:", result.is_success())
        # print("solution result:", result.get_solution_result())
        # print("solver:", result.get_solver_id().name())

        # # try:
        # #     details = result.get_solver_details()
        # #     print("solver details:", details)

        # #     for field in [
        # #         "rescode",
        # #         "solution_status",
        # #         "optimizer_time",
        # #     ]:
        # #         if hasattr(details, field):
        # #             print(f"{field}:", getattr(details, field))
        # # except Exception as e:
        # #     print("Could not read solver details:", e)

        # print("------------------\n")
        
        return traj, result        
        
    
    def _find_minimum_grasp_path(
            self,
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray, 
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:
            
        for i in range(self.max_grasps):
            if i % 10 == 0 and i > 0:
                print(f"Trying with {i} grasps")
            path = self._solve_for_n_grasps_CC(
                i,
                x_init,
                x_goal,
                P,
                G,
                lower_joint_limits,
                upper_joint_limits,
                c_free_polytopes
            )
            
            if path is not None:
                print(f"Found a solution with {i} grasps")
                return path  
        print(f"No solution found for less than {self.max_grasps} grasps")
    
    
    def _solve_for_n_grasps_CC(
            self,
            n_grasps: int, 
            x_init: np.ndarray,
            x_goal: np.ndarray,
            P: HPolyhedron,
            G: HPolyhedron,
            lower_joint_limits: np.ndarray, 
            upper_joint_limits: np.ndarray,
            c_free_polytopes: list[HPolyhedron]
            ) -> np.ndarray:
        # Initialize the program
        prog = MathematicalProgram()
        n_points = 2 * n_grasps + 2 # each grasp has a grasp and a release point, plus the initial and goal points
        n_vars = self.q_dim * n_points # for each point of q_dim length, we have n_points*q_dim variables
        non_gripper_dim = self.q_dim - self.gripper_dim # number of non-gripper variables per point
        
        # Create decision variables
        x = prog.NewContinuousVariables(n_vars, "x")
        
        # 1. Cost function: Minimize sum of squared distances between consecutive points
        """ ||x_{i+1} - x_{i}||^2  equiv || Cx - c ||^2  == 0 """
        
        c = np.zeros((n_vars - self.q_dim, 1)) 
        C = np.zeros((n_vars - self.q_dim, n_vars))
        C[:, self.q_dim:] = np.eye(n_vars - self.q_dim)
        C[:, :-self.q_dim] -= np.eye(n_vars - self.q_dim)
        
        Q, b = self._to_quadratic_form(C, c)
        Q += np.eye(Q.shape[1]) * 1e-6  # Add a small value to the diagonal to make Q positive definite
        
        prog.AddQuadraticCost(Q=Q, b=b, vars=x)
        
        # 2. Initial and goal constraints
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_init.flatten(), x[:self.q_dim])
        prog.AddLinearEqualityConstraint(np.eye(self.q_dim), x_goal.flatten(), x[-self.q_dim:])
        
        
        ###### Placement and Grasping Constraints should be adapted task specific #####
        
        # 3. Placement constraints (equality) -> Cap remains constant in transit paths
        """x_cap_{i+1} - x_cap_{i} == 0 for all even i, odd i+1"""
        for i in range(n_grasps + 1):
            idx = 2*self.q_dim*i # points to first variable of x_i for even i's
            idx1_cap = idx + self.q_dim - 1 # points to last variable of x_i (cap orientation)
            idx2_cap = idx1_cap + self.q_dim # points to last variable of x_{i+1} (cap orientation)
            prog.AddLinearEqualityConstraint(x[idx2_cap] - x[idx1_cap] == 0)
        
        # 4. Grasp constraints (equality between gripper and cap orientations)
        """x_cap_i - x_wrist_i - x_cap_{i+1} + x_wrist_{i+1} == 0 for odd i, even i+1"""
        for i in range(n_grasps): 
            
            idxp1 = 2*self.q_dim*(i+1) # this idx points to first variable of x_i+1
            idx   = idxp1 - self.q_dim # this idx points to first variable of x_i
            
            idxp1_wrist = idxp1 + non_gripper_dim
            idxp1_cap = idxp1 + self.q_dim - 1
            idx_wrist = idx + non_gripper_dim
            idx_cap = idx + self.q_dim - 1 # equiv to idxp1 - 1
            
            prog.AddLinearEqualityConstraint(x[idx_cap] - x[idx_wrist] - x[idxp1_cap] + x[idxp1_wrist] == 0)
        
        ###### end of manual constraints
        
        
        # 5. Inequality Constraints (Placement hull)
        for i in range(n_points):
            prog.AddLinearConstraint(
                P.A(),  # Coefficient matrix
                -np.inf * np.ones_like(P.b()),  # Lower bound
                P.b(),  # Upper bound
                x[self.q_dim*i:self.q_dim*(i+1)]
            )
        # 6. Inequality Constraints (Grasp hull) 
        # for i in range(n_points - 2):
        #     prog.AddLinearConstraint(
        #         G.A(),
        #         -np.inf * np.ones_like(G.b()),
        #         G.b(),
        #         x[self.q_dim*(i+1):self.q_dim*(i+2)]
        #     )
        
        # 7. Joint limits (inequality)
        prog.AddBoundingBoxConstraint(
            np.tile(lower_joint_limits, n_points),
            np.tile(upper_joint_limits, n_points),
            x
        )
        
        # 7. Collision-free polytope constraints (MIP)
        # Ensure all points are contained in at least one polytope
        
        M = 1e6  # Big-M constant (adjust based on problem scale)
        for i in range(n_points):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]
            
            # Creates one bineary 0-1 variable for each polytope
            z_i = prog.NewBinaryVariables(len(c_free_polytopes), name=f"z_{i}")
            
            # Each point x_i must be contained in at least one polytope
            prog.AddLinearConstraint(sum(z_i) >= 1)
            
            for j, poly in enumerate(c_free_polytopes):
                A_j = poly.A()
                b_j = poly.b()
                z_j = z_i[j]
                
                # Add constraints for each row of the polytope
                for k in range(A_j.shape[0]):
                    # Construct coefficient matrix [A_j_row | M]
                    coeffs = np.hstack([A_j[k], M])
                    # Combine x_i and z_j into variable vector
                    vars = np.hstack([x_i, [z_j]])
                    # A_j @ x_i + M * z[j] <= b_j + M
                    prog.AddLinearConstraint(
                        coeffs,
                        -np.inf,
                        b_j[k] + M,
                        vars
                    )
                
        # 8. Connected components constraints
        connected_components = self._compute_connected_components(P, G, c_free_polytopes)
        K = len(connected_components)
        
        # create binary variables for component membership (excluding init ang goal)
        z = {}
        for i in range (1, n_points-1): # exclude init and goal
            z[i] = prog.NewBinaryVariables(K, f"z_{i}")   
            # Each point must belong to exactly one component
            prog.AddLinearConstraint(sum(z[i]) == 1)
        
        # Consecutive points must belong to the same component, ignoring init and goal
        for i in range(1, n_points-2, 2):
            for k in range(K):
                prog.AddLinearConstraint(z[i][k] == z[i+1][k])
        
        # Enforce that each component contains at least one point
        for i in range(1, n_points-1):
            x_i = x[self.q_dim*i:self.q_dim*(i+1)]
            z_i = z[i]
            
            for k, comp in enumerate(connected_components):
                A_k = comp.A()
                b_k = comp.b()
                z_k = z_i[k]
                
                for j in range(A_k.shape[0]):
                    coeffs = np.hstack([A_k[j], M])
                    vars = np.hstack([x_i, [z_k]])
                    prog.AddLinearConstraint(coeffs, -np.inf, b_k[j] + M, vars)
        
        # Solve the problem
        solver = MosekSolver()
        result = solver.Solve(prog)
        
        if not result.is_success():
            return None
        
        return result.GetSolution(x).reshape(-1, self.q_dim)
    
    @staticmethod
    def _to_quadratic_form(C, c):
        return np.dot(C.T, C), -np.dot(C.T, c).flatten()
    
    def _compute_connected_components(
        self,
        placement: HPolyhedron,
        grasp: HPolyhedron,
        c_free: list[HPolyhedron]
    ) -> list[HPolyhedron]:
        """
        Computes convex connected components as:
        {placement ∩ grasp ∩ (union of c_free)}.
        Returns convex hulls of connected regions.
        """
        components = []
        
        # Compute intersections with each c_free polytope
        for poly in c_free:
            intersection = placement.Intersection(grasp).Intersection(poly)
            if not intersection.IsEmpty():
                components.append(intersection)
        
        # Merge overlapping components
        merged = []
        for comp in components:
            add_to_merged = False
            for m in merged:
                if comp.Intersection(m).IsEmpty():
                    continue
                # Merge overlapping components
                vertices_comp = visualizer.get_polytope_vertices(comp)
                vertices_m = visualizer.get_polytope_vertices(m)
                vertices = np.vstack([vertices_comp, vertices_m])
                merged_hull = ConvexHull(vertices)
                merged_poly = convex_hull_to_hpolyhedron(merged_hull)
                merged.remove(m)
                merged.append(merged_poly)
                add_to_merged = True
                break
            if not add_to_merged:
                merged.append(comp)
        
        return merged
    
    

    def _generate_cfree(self, irisalg: str = "irisnp") -> list[HPolyhedron]:
        if irisalg not in ["irisnp", "irisnp2", "iriszo"]:
            raise ValueError(f"Invalid irisalg: {irisalg}. Must be one of ['irisnp', 'irisnp2', 'iriszo']")
        
        generator = RandomGenerator(1234)
        checker = SceneGraphCollisionChecker(
            model=self.diagram,
            robot_model_instances=self.model_instances,
            edge_step_size=0.01,
        )
        
        parallelism = Parallelism(32)
        
        if irisalg != "irisnp":
            # For iriszo and irisnp2, we need to set the sampled iris options
            common_sampled_iris_options = CommonSampledIrisOptions()
            common_sampled_iris_options.delta = 0.05
            common_sampled_iris_options.epsilon = 0.01
            common_sampled_iris_options.max_iterations = 10
            common_sampled_iris_options.parallelism = parallelism
            common_sampled_iris_options.verbose = True
            common_sampled_iris_options.configuration_space_margin = 1e-5
            common_sampled_iris_options.termination_threshold = 1e-5
            common_sampled_iris_options.relative_termination_threshold = 1e-4
            common_sampled_iris_options.remove_all_collisions_possible = False
        
        options = IrisFromCliqueCoverOptions()
        options.num_points_per_visibility_round = 200
        options.coverage_termination_threshold = 0.99
        options.parallelism = parallelism
        
        if irisalg == "iriszo":
            iris_zo_options = IrisZoOptions()
            iris_zo_options.bisection_steps = 30
            iris_zo_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_zo_options
        elif irisalg == "irisnp2":
            iris_np2_options = IrisNp2Options()
            iris_np2_options.sampled_iris_options = common_sampled_iris_options
            options.iris_options = iris_np2_options
        else:
            # irisnp uses the default IrisOptions object on options.iris_options.
            options.iris_options.configuration_space_margin = 1e-5
        
        # See https://github.com/RobotLocomotion/drake/issues/21343
        regions = IrisInConfigurationSpaceFromCliqueCover(checker, options, generator, [])
        print("\nnumber of regions: ", len(regions))
        return regions
    
    

In [ ]:
model_instances = [plant.GetModelInstanceByName("panda"), plant.GetModelInstanceByName("panda_hand"), plant.GetModelInstanceByName("bottle_cap")]



# regions_dict_148 = LoadIrisRegionsYamlFile(
#     "cfree_drake_1_48.yaml"
# )
# regions_dict_149 = LoadIrisRegionsYamlFile(
#     "cfree_drake_1_49.yaml"
# )

# old_regions = list(regions_dict_148.values())
# new_regions = list(regions_dict_149.values())


planner = ManipulationPlanner(visualizer, placement_polytope, grasp_polytope, 50, model_instances=model_instances, gripper_dim=4, irisalg="iriszo")

INFO:drake:Allocating contexts to support implicit context parallelism 32
INFO:drake:Current Fraction of Domain Covered = 0
INFO:drake:IrisFromCliqueCover Iteration 1/100
INFO:drake:IrisZo using 32 threads.
INFO:drake:IrisZo finding region that is 0.01 collision free with 0.95 certainty using up to 11670 particles.
INFO:drake:IrisZo worst case test requires 11670 samples.
INFO:drake:IrisZo outer iteration 0
INFO:drake:IrisZo N_test 3193, N_col 246, thresh 15.965
INFO:drake:IrisZo probabilistic test failed! Continuing to compute hyperplanes.
INFO:drake:SeparatingPlanes iteration: 1 faces: 30
INFO:drake:IrisZo N_test 4302, N_col 201, thresh 21.51
INFO:drake:IrisZo probabilistic test failed! Continuing to compute hyperplanes.
INFO:drake:IrisZo N_test 4951, N_col 141, thresh 24.755
INFO:drake:IrisZo probabilistic test failed! Continuing to compute hyperplanes.
INFO:drake:IrisZo N_test 5411, N_col 106, thresh 27.055
INFO:drake:IrisZo probabilistic test failed! Continuing to compute hyperpla

## Walk around free polytope

In [ ]:
def walk_through_free_space(planner, verbose=True):
    """ Randomly move the robot through the free polytopes to visualize the free space in meshcat. This is done through IK, selecting a random linear velocity for the end effector and solving for the next configuration. This will happen for each polytope in the free space. The robot will move through the free space until it reaches the last polytope. 
    
    verbose: If True, prints the current polytope index out of the total number of polytopes and the current configuration of the robot.
    """
    
    for i, poly in enumerate(planner.cs_free):
        if verbose:
            print(f"Moving through polytope {i+1}/{len(planner.cs_free)}")
        # Get the vertices of the polytope
        vertices = visualizer.get_polytope_vertices(poly)
        # Select a random vertex
        random_vertex = vertices[np.random.choice(vertices.shape[0])]
        # Solve IK to move the robot to the random vertex
        q_sol, res = solve_ik_place_frame_at_pose(
            plant=planner.plant,
            plant_context=planner.plant_context,
            frame_B=planner.plant.GetFrameByName("panda_hand", planner.model_instances[0]),
            X_WG=RigidTransform(RotationMatrix(), random_vertex),
            pos_tol=0.01,
            theta_tol=np.deg2rad(5),
            min_distance=None,
            q_seed=planner.plant.GetPositions(planner.plant_context)
        )
        if q_sol is not None:
            
            # for 5 seconds, move the robot around the polytope, and bounce off the walls of the polytope, while staying inside the polytope. This is done by selecting a random linear velocity for the end effector and solving for the next configuration using IK. If the next configuration is outside the polytope, select a new random linear velocity and try again. Repeat this process for 5 seconds.
            start_time = time.time()
            while time.time() - start_time < 5:
                # Get the current position of the end effector
                X_WE = planner.plant.CalcRelativeTransform(planner.plant_context, planner.plant.world_frame(), planner.plant.GetFrameByName("panda_hand", planner.model_instances[0]))
                # Select a random linear velocity for the end effector
                random_velocity = np.random.uniform(-0.01, 0.01, size=3)
                # Compute the next position of the end effector
                next_position = X_WE.translation() + random_velocity
                # Check if the next position is inside the polytope
                if poly.PointInSet(next_position):
                    # Solve IK to move the robot to the next position
                    q_sol, res = solve_ik_place_frame_at_pose(
                        plant=planner.plant,
                        plant_context=planner.plant_context,
                        frame_B=planner.plant.GetFrameByName("panda_hand", planner.model_instances[0]),
                        X_WG=RigidTransform(RotationMatrix(), next_position),
                        pos_tol=0.01,
                        theta_tol=np.deg2rad(5),
                        min_distance=None,
                        q_seed=planner.plant.GetPositions(planner.plant_context)
                    )
                    if q_sol is not None:
                        planner.plant.SetPositions(planner.plant_context, q_sol)
                        planner.diagram.ForcedPublish(planner.diagram_context)
                        
                        
        else:
            if verbose:
                print(f"IK failed for polytope {i+1}/{len(planner.cs_free)}")
    
    

In [ ]:
# planner.visualize_cspace(factor=1, num_points=30, filled_polytopes=planner.cs_free)

In [ ]:
from pydrake.geometry.optimization import LoadIrisRegionsYamlFile
import numpy as np

path_148 = np.load(
    "path_drake_1_48.npy"
)

# traj = planner._generate_trajectory(path_148)

regions_dict_148 = LoadIrisRegionsYamlFile(
    "cfree_drake_1_48.yaml"
)
regions_dict_149 = LoadIrisRegionsYamlFile(
    "cfree_drake_1_49.yaml"
)

old_regions = list(regions_dict_148.values())
new_regions = list(regions_dict_149.values())

print("Number of free regions loaded:", len(old_regions))
print("Number of regions in the planner's c_free:", len(new_regions))

# planner.visualize_cspace(factor=1, num_points=30, filled_polytopes=free_regions)


In [ ]:
# print("Assert old and new free regions are the same or very similar")

# assert len(old_regions) == len(new_regions), "The number of free regions is different"


# old_vertices_list = []
# new_vertices_list = []

# for i, (old_region, new_region) in enumerate(zip(old_regions, new_regions)):
#     old_vertices = visualizer.get_polytope_vertices(old_region)
#     new_vertices = visualizer.get_polytope_vertices(new_region)
    
#     # list of old a new vertices
#     old_vertices_list.append(old_vertices)
#     new_vertices_list.append(new_vertices)
    
# # sort the vertices by the first coordinate of their shape to ensure consistent ordering for comparison
# old_vertices_list_sorted = sorted(old_vertices_list, key=lambda v: v.shape[0])
# new_vertices_list_sorted = sorted(new_vertices_list, key=lambda v: v.shape[0])

# # Compare the sorted vertices of each region
# for i, (old_vertices, new_vertices) in enumerate(zip(old_vertices_list_sorted, new_vertices_list_sorted)):
    
#     assert old_vertices.shape == new_vertices.shape, f"Region {i} vertices have different shapes: {old_vertices.shape} vs {new_vertices.shape}"
    
#     if not np.allclose(old_vertices, new_vertices, atol=1e-5):
#         print(f"Region {i} vertices are different.")
#         print("Old vertices:\n", old_vertices)
#         print("New vertices:\n", new_vertices)
#     else:
#         print(f"Region {i} vertices are the same or very similar.")


# q0 = path_148[0]
# q1 = path_148[1]

# def max_region_violation(region, q):
#     """
#     For A q <= b:
#         <= 0  means inside
#         > 0   means outside
#     """
#     q = np.asarray(q).reshape(-1)
#     return float(np.max(region.A() @ q - region.b()))


# q0_violations_old = np.array([
#     max_region_violation(region, q0)
#     for region in old_regions
# ])

# q0_violations_new = np.array([
#     max_region_violation(region, q0)
#     for region in new_regions
# ])

# q1_violations_old = np.array([
#     max_region_violation(region, q1)
#     for region in old_regions
# ])
# q1_violations_new = np.array([
#     max_region_violation(region, q1)
#     for region in new_regions
# ])


# print("q0 =", q0)
# print("q1 =", q1)



# print("\nBest q0 regions:")
# for i, j in zip(np.argsort(q0_violations_old)[:5], np.argsort(q0_violations_new)[:5]):
#     print(
#         f"  (old) region {i:2d}: "
#         f"max(Aq-b) = {q0_violations_old[i]:+.16e}"
#     )
#     print(
#         f"  (new) region {j:2d}: "
#         f"max(Aq-b) = {q0_violations_new[j]:+.16e}"
#     )

# print("\nBest q1 regions:")
# for i, j in zip(np.argsort(q1_violations_old)[:5], np.argsort(q1_violations_new)[:5]):
#     print(
#         f"  (old) region {i:2d}: "
#         f"max(Aq-b) = {q1_violations_old[i]:+.16e}"
#     )
#     print(
#         f"  (new) region {j:2d}: "
#         f"max(Aq-b) = {q1_violations_new[j]:+.16e}"
#     )

In [ ]:
# planner.cs_free = old_regions

q0 = path_148[0]
q1 = path_148[1]


# for weight in [
#     1e-6,
#     1e-5,
#     1e-4,
#     1e-3,
#     1e-2,
#     1e-1,
#     1.0,
#     10.0,
# ]:
#     print("\n")
#     print("=" * 70)
#     print("PATH LENGTH WEIGHT:", weight)
#     print("=" * 70)

traj, result = planner._generate_edge_trajectory(
    q0,
    q1
)

print(
    "FINAL RESULT:",
    result.get_solution_result(),
)

In [ ]:
    # x_init = np.array([-3.14, -0.78, -0.03]) # [cap angle, gripper angle, finger translation]
    # x_goal = np.array([3.14, -0.78, -0.03])

# switch to gripper angle, finger translation, cap angle for planner
x_init = np.array([-0.72, -0.03, -3.14]) # [gripper angle, finger translation, cap angle]
x_goal = np.array([-0.72, -0.03, 3.14])

print(visualizer.check_collision_q_by_ik(x_init))
print(visualizer.check_collision_q_by_ik(x_goal))
path, traj = planner.compute_trajectory(x_init, x_goal)

In [ ]:
planner.plant.SetPositions(planner.plant_context, x_init)
traj_path = planner.display_trajectory(traj, plotly=True)


In [ ]:
# from pydrake.geometry.optimization import SaveIrisRegionsYamlFile

# SaveIrisRegionsYamlFile(
#     "cfree_drake_1_48.yaml",
#     {
#         f"region_{i:03d}": region
#         for i, region in enumerate(planner.cs_free)
#     },
# )

# np.save(
#     "path_drake_1_49.npy",
#     path,
# )